![](https://github.com/destination-earth/DestinE-DataLake-Lab/blob/main/img/DestinE-banner.jpg?raw=true)

# Extraction of ClimateDT data
Data portfolio: https://confluence.ecmwf.int/display/DDCZ/Climate+DT+Phase+1+data+catalogue#ClimateDTPhase1datacatalogue-Fieldsonasinglelevelorsurface


Requirements:

- params: 228/167
- temporal extent: 1990-2020
- spatial: Austria

- variables
  - temperature
    - only specific time stamps
  - precipitation
    - houerly data needed  

based on these we can only extract data from IFS-NEMO CMIP6 corresponding to scenario index 0

In [1]:
# all imnports needed to run the notebook
from typing import Literal, Dict
import pandas as pd
from pathlib import Path
import s3fs

import earthkit.geo.cartography
import concurrent.futures
import xarray as xr
import os

In [2]:
%%capture cap
%run ./src/desp-authentication.py

In [3]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]
access_token

'Token successfully written to /home/koenifra/.polytopeapirc'

In [4]:
scenarios = pd.read_csv("./climate-dt-scenarios.csv", sep=";")
scenarios

,params,model,level type,experiment,resolution,temporal extent,activity,typeOfSimulation
0,167/260048,ICON,sfc,hist,high,1990-2024,CMIP6,NaN
1,141/167/228/260048,IFS-NEMO,sfc,hist,high,1990-2024,CMIP6,NaN
2,228141,IFS-NEMO,sol,hist,high,1990-2024,CMIP6,NaN
3,167/228,IFS-FESOM,sfc,cont,high,1990-2004,HighResMIP,Control simulation
4,167/228,IFS-NEMO,sfc,cont,high,1990-2007,HighResMIP,Control simulation
5,167/260048,ICON,sfc,SSP3-7.0,high,2020-2039,ScenarioMIP,Future projection
6,167/228,IFS-FESOM,sfc,SSP3-7.0,high,2020-2039,ScenarioMIP,Future projection
7,141/167/228/260048,IFS-NEMO,sfc,SSP3-7.0,high,2020-2039,ScenarioMIP,Future projection
8,228141,IFS-NEMO,sol,SSP3-7.0,high,2020-2050,ScenarioMIP,NaN
9,167/228,IFS-FESOM,sfc,cont,high,2017-2023,story-nudging,Storyline simulation


**Only scenario with index 0 will be considered hereafter, IFS-NEMO | sfc | 1990-2024 | 167/228**

In [5]:
scenario = scenarios.iloc[3]
scenario

params                         167/228
model                        IFS-FESOM
level type                         sfc
experiment                        cont
resolution                        high
temporal extent              1990-2004
activity                    HighResMIP
typeOfSimulation    Control simulation
Name: 3, dtype: object

In [6]:
def get_request_dict(experiment: str,
                      activity: str,
                      level_type: str,
                      datestring: str,
                      model: str,
                      parameter: str,
                      location: list,
                      feature: Literal["timeseries", "polygon"]="timeseries",
                    #   time_resolution: str="0000/to/2300",
                    time_resolution: str="0000",
                      resolution: str="high") -> Dict:
    # request time-series data
    if feature == "timeseries":
        feature_dict = {
            "type" : "timeseries",
            "points": location,
            "time_axis": "date"
        }
    elif feature == "polygon":
        feature_dict = {
            "type" : "polygon",
            "shape": location
        }
    else:
        raise TypeError("feature not supported")
    
    request = {
        # static parameters of climate dt data
        "class": "d1",
        "dataset": "climate-dt",
        "generation": "1",
        # "expver": "0001",
        "expver": "1",
        "stream": "clte",
        "type": "fc",
        # generic
        "activity": activity,
        "experiment": experiment,
        "levtype": level_type,
        "date": datestring,
        "model": model,
        "param": parameter,
        "realization": "1",
        "resolution": resolution,
        "time": time_resolution,
        "feature": feature_dict
    }

    # commented out to check if only one level is request it gets faster or not
    if level_type == "sol":
        # request["levelist"] = "1/to/5"
        request["levelist"] = "1"

    # return the request dict
    return request    
    

In [7]:
def extract_dtdata(request_dict: dict,
                   bucketname: str="",
                   dt_url: str="polytope.lumi.apps.dte.destination-earth.eu"):
    import earthkit.data as ekd
   
    try:
        # get covjson
        ds = ekd.from_source("polytope", 
                             "destination-earth", 
                             request_dict,
                             stream=False, 
                             address=dt_url)
    except Exception as e:
        print(e)        
        
    return ds.to_xarray()

In [8]:
def get_datestring(temp_extent: str):
    '''
    Example output string "20200101/to/20210101"
    '''
    years = temp_extent.split("-")
    return f"{years[0]}0101/to/{years[1]}1231"
    

## Get polygon for Austria

In [9]:
countries = ["Austria"]
polygon_at = earthkit.geo.cartography.country_polygons(countries, resolution=50e6)

In [10]:
import json
polygon_alps = "[[[43.000000, 4.000000], [43.000000, 18.000000], [50.000000, 18.000000], [50.000000, 4.000000], [43.000000, 4.000000]]]"

# Parse the string into a nested list
polygon_alps = json.loads(polygon_alps)

polygon_alps

[[[43.0, 4.0], [43.0, 18.0], [50.0, 18.0], [50.0, 4.0], [43.0, 4.0]]]

## Extract TS of data over Austria and the given scenario

In [11]:
#Supress default INFO logging

import logging
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

In [12]:
def extract_austria_ts(scenario: dict, datestr: str, params: str, polygon: list):
    request_dict = get_request_dict(scenario["experiment"],
                        scenario["activity"],
                        scenario["level type"],
                        datestr,
                        scenario["model"],
                        params,
                        polygon,
                        "polygon")
    return extract_dtdata(request_dict)

In [13]:
def get_point_data(point, scenario):
    latlon = [[float(point["latitude"]),float(point["longitude"])]] 
    
    ts = get_request_dict(scenario["experiment"],
                        scenario["activity"],
                        scenario["level type"],
                        get_datestring(scenario["temporal extent"]),
                        scenario["model"],
                        scenario["params"],
                        latlon)
    return ts.assign_coords({"stationid": geosphere_station["id"]}) \
        .expand_dims(dim="stationid")

Extract one day of data to get all points within Austria

In [14]:
# ds = extract_austria_ts(scenario, "19950101", params="141", polygon=polygon_at)
ds = extract_austria_ts(scenario, "20020101", params="167", polygon=polygon_at)

2026-05-04 14:04:43 - INFO - Key read from /home/koenifra/.polytopeapirc
2026-05-04 14:04:43 - INFO - Sending request...
{'request': 'activity: HighResMIP\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20020101'\n"
            'experiment: cont\n'
            "expver: '1'\n"
            'feature:\n'
            '  shape:\n'
            '  - - - 47.270751953125\n'
            '      - 9.527539062500011\n'
            '    - - 47.391796875\n'
            '      - 9.609082031250011\n'
            '    - - 47.467041015625\n'
            '      - 9.625878906250023\n'
            '    - - 47.511132812499994\n'
            '      - 9.554394531250011\n'
            '    - - 47.524218749999996\n'
            '      - 9.524023437500006\n'
            '    - - 47.534033203125\n'
            '      - 9.548925781250006\n'
            '    - - 47.52587890625\n'
            '      - 9.650585937500011\n'
            '    - - 47.55078125\n'
            '      - 9.7

Polytope error
Situation: trying to download data
Description: HTTP CLIENT ERROR (400)
URL: https://polytope.lumi.apps.dte.destination-earth.eu:443/api/v1/requests/427f237d-d3b5-4090-957b-73c86cf2585c
HTTP method: GET
Request header/body contents:
{'headers': {'Authorization': 'Bearer **********HhbQ'}, 'json': None}
Expected responses: 200, 202
Received response: CLIENT ERROR (400)
Details:
Request failed with error:
Matched datasource polytope-climate

Polytope Feature Extraction Error: No data for {'activity': 'highresmip', 'class': 'd1', 'dataset': 'climate-dt', 'date': '20020101', 'experiment': 'cont', 'expver': '0001', 'generation': '1', 'levtype': 'sfc', 'model': 'ifs-fesom', 'param': '167', 'realization': '1', 'resolution': 'high', 'stream': 'clte', 'type': 'fc'} is available on the FDB.


UnboundLocalError: cannot access local variable 'ds' where it is not associated with a value

In [43]:
print(ds)

<xarray.Dataset> Size: 827kB
Dimensions:    (datetimes: 1, number: 1, steps: 1, points: 20668)
Coordinates:
  * datetimes  (datetimes) <U20 80B '2022-01-01 00:00:00Z'
  * number     (number) int64 8B 0
  * steps      (steps) int64 8B 0
  * points     (points) int64 165kB 0 1 2 3 4 ... 20663 20664 20665 20666 20667
    latitude   (points) float64 165kB 43.01 43.01 43.01 ... 49.99 49.99 49.99
    longitude  (points) float64 165kB 4.005 4.095 4.185 ... 17.78 17.88 17.99
    levelist   (points) float64 165kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Data variables:
    sd         (datetimes, number, steps, points) float64 165kB 0.0 ... 1.621...
Attributes: (12/15)
    activity:     scenariomip
    class:        d1
    dataset:      climate-dt
    experiment:   ssp3-7.0
    expver:       0001
    generation:   1
    ...           ...
    resolution:   high
    stream:       clte
    type:         fc
    number:       0
    step:         0
    date:         2022-01-01 00:00:00Z


In [44]:
npoints_austria = ds["points"].size
npoints_austria

20668

In [45]:
all_points = ds
all_points.drop_dims("number")

<xarray.Dataset> Size: 661kB
Dimensions:    (datetimes: 1, steps: 1, points: 20668)
Coordinates:
  * datetimes  (datetimes) <U20 80B '2022-01-01 00:00:00Z'
  * steps      (steps) int64 8B 0
  * points     (points) int64 165kB 0 1 2 3 4 ... 20663 20664 20665 20666 20667
    latitude   (points) float64 165kB 43.01 43.01 43.01 ... 49.99 49.99 49.99
    longitude  (points) float64 165kB 4.005 4.095 4.185 ... 17.78 17.88 17.99
    levelist   (points) float64 165kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Data variables:
    *empty*
Attributes: (12/15)
    activity:     scenariomip
    class:        d1
    dataset:      climate-dt
    experiment:   ssp3-7.0
    expver:       0001
    generation:   1
    ...           ...
    resolution:   high
    stream:       clte
    type:         fc
    number:       0
    step:         0
    date:         2022-01-01 00:00:00Z

In [46]:
all_points

<xarray.Dataset> Size: 827kB
Dimensions:    (datetimes: 1, number: 1, steps: 1, points: 20668)
Coordinates:
  * datetimes  (datetimes) <U20 80B '2022-01-01 00:00:00Z'
  * number     (number) int64 8B 0
  * steps      (steps) int64 8B 0
  * points     (points) int64 165kB 0 1 2 3 4 ... 20663 20664 20665 20666 20667
    latitude   (points) float64 165kB 43.01 43.01 43.01 ... 49.99 49.99 49.99
    longitude  (points) float64 165kB 4.005 4.095 4.185 ... 17.78 17.88 17.99
    levelist   (points) float64 165kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Data variables:
    sd         (datetimes, number, steps, points) float64 165kB 0.0 ... 1.621...
Attributes: (12/15)
    activity:     scenariomip
    class:        d1
    dataset:      climate-dt
    experiment:   ssp3-7.0
    expver:       0001
    generation:   1
    ...           ...
    resolution:   high
    stream:       clte
    type:         fc
    number:       0
    step:         0
    date:         2022-01-01 00:00:00Z

In [47]:
all_points.to_dataframe().to_csv("points_IFS_NEMO_ScenarioMIP_141.csv")

In [42]:
from datetime import datetime
start, end = scenario["temporal extent"].split("-")
ndays = (datetime(int(end)+1,1,1) - datetime(int(start),1,1)).days
ntimestamps = 24 * ndays
ntimestamps

306816

## Write zarr store which gets finally populated with data

In [43]:
import pandas as pd

# Define start and end dates
start = "1990-01-01"
end = "2025-12-01"

# Generate monthly start dates
dates = pd.date_range(start=start, end=end, freq='MS')  # 'MS' = Month Start

# Build series of formatted date strings
date_strings = pd.Series([
    f"{d.strftime('%Y%m%d')}/to/{(d + pd.offsets.MonthBegin(1)).strftime('%Y%m%d')}"
    for d in dates
])

print(date_strings)

0      19900101/to/19900201
1      19900201/to/19900301
2      19900301/to/19900401
3      19900401/to/19900501
4      19900501/to/19900601
               ...         
427    20250801/to/20250901
428    20250901/to/20251001
429    20251001/to/20251101
430    20251101/to/20251201
431    20251201/to/20260101
Length: 432, dtype: object


In [50]:
def get_datestring(temp_extent: str):
    '''
    Example output string "20200101/to/20210101"
    '''
    years = temp_extent.split("-")
    return f"{years[0]}0101/to/{years[1]}1231"


print(get_datestring(scenario["temporal extent"]))

19900101/to/20241231


In [44]:
import earthkit
n_down_threads = 10
earthkit.data.config.set("number-of-download-threads", n_down_threads)
earthkit.data.config

Name,Value,Default
cache-policy,'off','off'
check-out-of-date-urls,True,True
download-out-of-date-urls,False,False
grib-field-policy,'persistent','persistent'
grib-file-serialisation-policy,'path','path'
grib-handle-cache-size,1,1
grib-handle-policy,'cache','cache'
maximum-cache-disk-usage,'95%','95%'
maximum-cache-size,None,None
number-of-download-threads,10,5


In [28]:
# import s3fs
# from rich.prompt import Prompt

# bucket = "68e13833a1624f43ba2cac01376a18af:destine-climate-dt"
# prefix = ""
# s3_endpoint = "https://objects.eodc.eu"

# # Create the S3FileSystem with a custom endpoint
# s3_eodc = s3fs.S3FileSystem(
#     endpoint_url=s3_endpoint,
#     key=Prompt.ask(prompt="S3 Key"),
#     secret=Prompt.ask(prompt="S3 Secret", password=True),
#     use_ssl=True,
# )

S3 Key:

S3 Secret:

In [ ]:
eodc_s3 = s3fs.S3FileSystem(
    key="",
    secret="",
    client_kwargs={
        "endpoint_url": "https://objects.eodc.eu"
    })
# zarr_stores = eodc_s3.ls("destine-climate-dt")
# zarr_stores

In [46]:
eodc_s3.ls("destine-climate-dt")

['destine-climate-dt/ICON_sfc_CMIP6.zarr',
 'destine-climate-dt/ICON_sfc_ScenarioMIP.zarr',
 'destine-climate-dt/IFS-FESMO-cont-rechunked.zarr',
 'destine-climate-dt/IFS-NEMO-CMIP6-rechunked.zarr',
 'destine-climate-dt/IFS-NEMO-CMIP6-rechunkedTS.zarr',
 'destine-climate-dt/IFS-NEMO-CMIP6-ts.zarr',
 'destine-climate-dt/IFS-NEMO-down',
 'destine-climate-dt/IFS-NEMO-sfc-CMIP6-consolidated_rechunked.zarr',
 'destine-climate-dt/IFS-NEMO-sfc-CMIP6-consolidated_rechunked.zarr_tmp',
 'destine-climate-dt/IFS-NEMO_sfc_CMIP6.zarr',
 'destine-climate-dt/IFS-NEMO_sfc_ScenarioMIP.zarr',
 'destine-climate-dt/climate-dt.zarr']

In [47]:
eodc_s3.mkdir("destine-climate-dt/climatedt-austria-data")

In [51]:
store_path = f"{scenario['model']}_{scenario['level type']}_{scenario['activity']}_{scenario['experiment']}.zarr"
# store_path = f"{scenario['model']}_{scenario['level type']}_{scenario['activity']}_SSP3-7.zarr"
print(f"Data will be stored in: {store_path}")

Data will be stored in: IFS-NEMO_sfc_CMIP6_hist.zarr


In [ ]:
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = []
    for ind, p in all_points.iloc[8:].iterrows():
    #for station in station_list:
        futures.append(executor.submit(get_station_data, geosphere_station=station, scenario=scenario))
    
    for future in concurrent.futures.as_completed(futures):
        if not os.path.exists(store_path):
            #scenario_ts = future.result().copy()
            future.result().chunk(chunks={"stationid": 1,
                              "latitude": 1,
                              "longitude": 1,
                              "levelist": 1,
                              "number": 1,
                              "datetime": 1,
                              "t": "auto"
                             }).to_zarr(store=store_path,
                        mode="w")
        else:
            #scenario_ts = xr.concat([scenario_ts, future.result()], dim="stationid")
            future.result().chunk(chunks={"stationid": 1,
                              "latitude": 1,
                              "longitude": 1,
                              "levelist": 1,
                              "number": 1,
                              "datetime": 1,
                              "t": "auto"
                             }).to_zarr(store=store_path,
                        append_dim="stationid")

In [ ]:
import concurrent.futures
import xarray as xr
import os


store_path_zarr = f"destine-climate-dt/climatedt-austria-data/{store_path}"
store = eodc_s3.get_mapper(store_path_zarr)

with concurrent.futures.ThreadPoolExecutor(max_workers=n_down_threads) as executor:
    futures = []
    for ind, p in all_points.iterrows():
        # futures.append(executor.submit(extract_austria_ts, 
        #                                scenario=scenario, 
        #                                datestr=get_datestring(scenario["temporal extent"]),
        #                                params="167/228",
        #                                polygon=polygon_at))

        futures.append(executor.submit(extract_austria_ts, 
                                       scenario=scenario, 
                                       datestr=get_datestring(scenario["temporal extent"]),
                                       params="167/228",
                                       polygon=polygon_at))
         
    for future in concurrent.futures.as_completed(futures):
        ds = future.result().squeeze().chunk({"datetimes": -1, "points": 1})
        if not eodc_s3.exists(store_path_zarr):  # ✅ S3-aware check
            ds.to_zarr(store=store, mode="w")
        else:
            ds.to_zarr(store=store, append_dim="points", mode="a")

2026-04-13 13:05:07 - INFO - Key read from /home/koenifra/.polytopeapirc
2026-04-13 13:05:07 - INFO - Key read from /home/koenifra/.polytopeapirc
2026-04-13 13:05:07 - INFO - Sending request...
{'request': 'activity: CMIP6\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            'date: 19900101/to/20241231\n'
            'experiment: hist\n'
            "expver: '0001'\n"
            'feature:\n'
            '  shape:\n'
            '  - - - 47.270751953125\n'
            '      - 9.527539062500011\n'
            '    - - 47.391796875\n'
            '      - 9.609082031250011\n'
            '    - - 47.467041015625\n'
            '      - 9.625878906250023\n'
            '    - - 47.511132812499994\n'
            '      - 9.554394531250011\n'
            '    - - 47.524218749999996\n'
            '      - 9.524023437500006\n'
            '    - - 47.534033203125\n'
            '      - 9.548925781250006\n'
            '    - - 47.52587890625\n'
            '      - 

In [ ]:
import concurrent.futures
import xarray as xr
import os

with concurrent.futures.ThreadPoolExecutor(max_workers=n_down_threads) as executor:
    futures = []
    for ind, datestr in date_strings[0:2].items():
        futures.append(executor.submit(extract_austria_ts, 
                                       scenario=scenario, 
                                       datestr=datestr,
                                       params="167/228",
                                       polygon=polygon_at))

    counter=0
    for future in concurrent.futures.as_completed(futures):
        future.result().squeeze().chunk(
            chunks={"datetimes": -1, "points": -1}).to_zarr(
                store=s3_eodc.get_mapper(f"destine-climate-dt/climatedt-austria-data/{store_path}"),mode="w")
        counter += 1